In [1]:
# Setup
import os, sys
from dotenv import load_dotenv

PROJECT_ROOT = r"C:\Users\user\Downloads\SHL Assessment"

os.chdir(PROJECT_ROOT)

# Add every folder explicitly
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "scraper"))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "embeddings"))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "evaluation"))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "api"))

os.environ["CHROMA_DB_PATH"]   = os.path.join(PROJECT_ROOT, "data", "chroma_db")
os.environ["ASSESSMENTS_PATH"] = os.path.join(PROJECT_ROOT, "data", "assessments.json")

load_dotenv(os.path.join(PROJECT_ROOT, ".env"), override=True)

# Confirm all folders are in path
print("Paths added:")
for p in sys.path[:6]:
    print(" ", p)
print("Gemini key:", os.getenv("GEMINI_API_KEY", "NOT FOUND")[:8], "...")

Paths added:
  C:\Users\user\Downloads\SHL Assessment\api
  C:\Users\user\Downloads\SHL Assessment\evaluation
  C:\Users\user\Downloads\SHL Assessment\embeddings
  C:\Users\user\Downloads\SHL Assessment\scraper
  C:\Users\user\Downloads\SHL Assessment
  C:\Users\user
Gemini key: AIzaSyBa ...


In [2]:
# CELL 2: Scrape
from scrape_catalog import scrape_listing_pages, save

assessments = scrape_listing_pages()
save(assessments)
print(f"✅ {len(assessments)} assessments saved")

[1/2] Scraping catalog listing pages...
  Fetching page 1: https://www.shl.com/products/product-catalog/?start=0&type=1
  Total pages: 32  (~384 assessments)
  Page 2/32 ...
  Page 3/32 ...
  Page 4/32 ...
  Page 5/32 ...
  Page 6/32 ...
  Page 7/32 ...
  Page 8/32 ...
  Page 9/32 ...
  Page 10/32 ...
  Page 11/32 ...
  Page 12/32 ...
  Page 13/32 ...
  Page 14/32 ...
  Page 15/32 ...
  Page 16/32 ...
  Page 17/32 ...
  Page 18/32 ...
  Page 19/32 ...
  Page 20/32 ...
  Page 21/32 ...
  Page 22/32 ...
  Page 23/32 ...
  Page 24/32 ...
  Page 25/32 ...
  Page 26/32 ...
  Page 27/32 ...
  Page 28/32 ...
  Page 29/32 ...
  Page 30/32 ...
  Page 31/32 ...
  Page 32/32 ...
  Collected 377 Individual Test Solutions

Saved 377 assessments → C:\Users\user\Downloads\SHL Assessment\scraper\../data/assessments.json
  With description : 0/377
  With duration    : 0/377
  Remote testing   : 377
  Adaptive/IRT     : 37
  Test type dist.  : {'K': 240, 'P': 66, 'S': 43, 'A': 32, 'C': 19, 'B': 17, 'D':

In [3]:
# CELL 3: Build index 
from build_index import build_index
build_index()
print("✅ Index built")

Loading assessments from C:\Users\user\Downloads\SHL Assessment\embeddings\../data/assessments.json...
  Loaded 377 assessments
Loading sentence transformer model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Model loaded ✅
Setting up ChromaDB at C:\Users\user\Downloads\SHL Assessment\data\chroma_db...
  Deleted existing collection (rebuilding fresh)
Building embeddings and storing in ChromaDB...
  Encoding documents (this may take a minute)...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

  Stored batch 0–100
  Stored batch 100–200
  Stored batch 200–300
  Stored batch 300–377

✅ Index built with 377 assessments
   ChromaDB saved at: C:\Users\user\Downloads\SHL Assessment\data\chroma_db
✅ Index built


In [4]:
# Evaluate 
import importlib.util
import os

# Load evaluate.py directly from its path
eval_path = os.path.join(PROJECT_ROOT, "evaluation", "evaluate.py")
spec = importlib.util.spec_from_file_location("evaluate", eval_path)
evaluate_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(evaluate_module)

# Now call the function
score = evaluate_module.evaluate()
print(f"✅ Mean Recall@10 = {score:.4f}")

C:\Users\user\Downloads\SHL Assessment\api\recommender.py:10: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


SHL Recommender — Train Set Evaluation

Loading train data from dataset...
  10 labeled queries loaded

Loading recommender...
Initializing SHL Recommender...
  Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Connecting to ChromaDB...
  Loaded collection with 377 assessments ✅

[1/10] Query: I am hiring for Java developers who can also collaborate effectively w...
  Ground truth: 5 assessments

[Recommender] Query: I am hiring for Java developers who can also collaborate effectively with my bus...
  [1/4] Expanding query with LLM...
        Role: Java Developer, Types: ['K', 'P'], Max duration: 40 min
  [2/4] Searching vector index...
        Found 30 candidates
  [3/4] Applying duration filter...
        30 candidates after filter
  [4/4] Re-ranking for relevance and balance...
        Selected 10 final assessments
  Predicted: 10 assessments
  Recall@10: 0.600
  ✅ HITS: ['java-8-new', 'core-java-advanced-level-new', 'interpersonal-communications']
  ❌ MISSED: ['automata-fix-new', 'core-java-entry-level-new']

[2/10] Query: I want to hire new graduates for a sales role in my company, the budge...
  Ground truth: 9 assessments

[Recommender] Query: I want to hire new graduates for a sales

In [5]:
# Generate predictions CSV
import importlib.util
import os

pred_path = os.path.join(PROJECT_ROOT, "evaluation", "generate_predictions.py")
spec = importlib.util.spec_from_file_location("generate_predictions", pred_path)
pred_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pred_module)

pred_module.generate_predictions()
print("✅ test_predictions.csv ready")

Generating Test Set Predictions

Loaded 9 test queries
Loading recommender...

[1/9] Looking to hire mid-level professionals who are proficient in Python, SQL and Ja...

[Recommender] Query: Looking to hire mid-level professionals who are proficient in Python, SQL and Ja...
  [1/4] Expanding query with LLM...
        Role: Software Developer, Types: ['K', 'P'], Max duration: 60 min
  [2/4] Searching vector index...
        Found 30 candidates
  [3/4] Applying duration filter...
        30 candidates after filter
  [4/4] Re-ranking for relevance and balance...
  [WARN] LLM re-ranking failed: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 35.572

In [6]:
from generate_predictions import generate_predictions
generate_predictions()
print("✅ New CSV generated")

Generating Test Set Predictions

Loaded 9 test queries
Loading recommender...

[1/9] Looking to hire mid-level professionals who are proficient in Python, SQL and Ja...

[Recommender] Query: Looking to hire mid-level professionals who are proficient in Python, SQL and Ja...
  [1/4] Expanding query with LLM...
  [WARN] LLM expansion failed: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 18.062033105s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateR

In [22]:
import json, os, sys

# Load recommender directly
sys.path.insert(0, r"C:\Users\user\Downloads\SHL Assessment\api")
from recommender import get_recommender

rec = get_recommender()
results = rec.recommend("I need a Python developer", top_k=2)
response = rec.format_response(results)

# Pretty print
print(json.dumps(response, indent=2))


[Recommender] Query: I need a Python developer...
  [1/4] Expanding query with LLM...
  [WARN] LLM expansion failed: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 11.415055923s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds

In [49]:
import requests

BASE_URL = "https://aritrasarkar788-shl-recommender.hf.space"

# Test 1: Health check
r1 = requests.get(f"{BASE_URL}/health", timeout=60)
print("Health:", r1.json())

# Test 2: Recommend
r2 = requests.post(
    f"{BASE_URL}/recommend",
    json={"query": "I need a Python developer who can collaborate with teams"},
    timeout=120
)
data = r2.json()
print(f"\nAssessments returned: {len(data['recommended_assessments'])}")
for a in data['recommended_assessments']:
    print(f"  - {a['name']} | {a['test_type']} | {a['duration']} min")

Health: {'status': 'healthy'}

Assessments returned: 10
  - Programming Concepts | ['Knowledge & Skills'] | 0 min
  - Python (New) | ['Knowledge & Skills'] | 0 min
  - Smart Interview Live Coding | ['Knowledge & Skills'] | 0 min
  - Shell Scripting (New) | ['Knowledge & Skills'] | 0 min
  - Computer Science (New) | ['Knowledge & Skills'] | 0 min
  - Agile Software Development | ['Knowledge & Skills'] | 0 min
  - Global Skills Development Report | ['Ability & Aptitude', 'Assessment Exercises', 'Biodata & Situational Judgement', 'Competencies', 'Development & 360', 'Personality & Behaviour'] | 0 min
  - Networking and Implementation (New) | ['Knowledge & Skills'] | 0 min
  - Linux Operating System | ['Knowledge & Skills'] | 0 min
  - GIT (New) | ['Knowledge & Skills'] | 0 min


In [51]:
frontend_path = r"C:\Users\user\Downloads\SHL Assessment\frontend\index.html"

with open(frontend_path, "r", encoding="utf-8") as f:
    content = f.read()

# Replace localhost with HF URL
content = content.replace(
    "http://localhost:8000",
    "https://aritrasarkar788-shl-recommender.hf.space"
)

with open(frontend_path, "w", encoding="utf-8") as f:
    f.write(content)

print("✅ Frontend updated with live API URL")

✅ Frontend updated with live API URL


In [53]:
import requests
import os
import pandas as pd

API_URL      = "https://aritrasarkar788-shl-recommender.hf.space"
FRONTEND_URL = "https://shl-assessment-recommender-taupe.vercel.app"

print("=" * 55)
print("FINAL SUBMISSION CHECKLIST")
print("=" * 55)

# Check 1: Health endpoint
try:
    r = requests.get(f"{API_URL}/health", timeout=60)
    assert r.json() == {"status": "healthy"}
    print("✅ /health works:", r.json())
except Exception as e:
    print(f"❌ /health failed: {e}")

# Check 2: Recommend endpoint
try:
    r = requests.post(
        f"{API_URL}/recommend",
        json={"query": "Java developer with collaboration skills"},
        timeout=120
    )
    data = r.json()
    count = len(data["recommended_assessments"])
    assert 1 <= count <= 10
    print(f"✅ /recommend works: {count} assessments returned")
except Exception as e:
    print(f"❌ /recommend failed: {e}")

# Check 3: All required fields
try:
    first = data["recommended_assessments"][0]
    required = ["url","name","adaptive_support",
                "description","duration","remote_support","test_type"]
    for field in required:
        assert field in first
    print("✅ All required fields present")
except Exception as e:
    print(f"❌ Fields check failed: {e}")

# Check 4: test_type is full names
try:
    types = first["test_type"]
    assert all(len(t) > 1 for t in types)
    print(f"✅ test_type correct: {types}")
except Exception as e:
    print(f"❌ test_type check failed: {e}")

# Check 5: CSV
try:
    csv_path = r"C:\Users\user\Downloads\SHL Assessment\data\test_predictions.csv"
    df = pd.read_csv(csv_path)
    assert list(df.columns) == ["Query", "Assessment_url"]
    assert df["Query"].nunique() == 9
    print(f"✅ CSV ready: {len(df)} rows, {df['Query'].nunique()} queries")
except Exception as e:
    print(f"❌ CSV check failed: {e}")

# Check 6: Frontend accessible
try:
    r = requests.get(FRONTEND_URL, timeout=30)
    assert r.status_code == 200
    print(f"✅ Frontend accessible: {FRONTEND_URL}")
except Exception as e:
    print(f"❌ Frontend failed: {e}")

print()
print("=" * 55)
print("YOUR 5 SUBMISSION ITEMS")
print("=" * 55)
print(f"1. API URL      : {API_URL}")
print(f"2. Frontend URL : {FRONTEND_URL}")
print(f"3. GitHub URL   : https://github.com/Aritrasarkar788/SHL_ASSESSMENT_RECOMMENDER")
print(f"4. CSV file     : data/test_predictions.csv")
print(f"5. Approach doc : APPROACH.md")
print("=" * 55)

FINAL SUBMISSION CHECKLIST
✅ /health works: {'status': 'healthy'}
✅ /recommend works: 10 assessments returned
✅ All required fields present
✅ test_type correct: ['Competencies', 'Ability & Aptitude', 'Personality & Behaviour']
✅ CSV ready: 90 rows, 9 queries
✅ Frontend accessible: https://shl-assessment-recommender-taupe.vercel.app

YOUR 5 SUBMISSION ITEMS
1. API URL      : https://aritrasarkar788-shl-recommender.hf.space
2. Frontend URL : https://shl-assessment-recommender-taupe.vercel.app
3. GitHub URL   : https://github.com/Aritrasarkar788/SHL_ASSESSMENT_RECOMMENDER
4. CSV file     : data/test_predictions.csv
5. Approach doc : APPROACH.md
